# RAG11 Nutrition — Stage 1.9: Verify All Data

Standalone integrity check between the local chunk files
(`stage1_eda_output/source{1,2,3}/*.json`) and what's actually sitting in
`rag11_chunks_parent_table` / `rag11_chunks_child_table`. Previously this
was a small "row counts + one smoke-test query" cell at the end of
`stage1_2_eda_load_chunks.ipynb`; it's pulled out here and expanded into a
real per-record comparison, run independently whenever you want to confirm
a load actually landed correctly (including after a partial/checkpointed
run, or after re-tuning stage1_1 and reloading).

What it checks, for every single chunk file, not just row counts:
- the row exists in Supabase at all (nothing silently missing)
- its `rowJSON` matches the local file's content exactly (nothing corrupted
  or stale from a previous run with different section boundaries)
- its `rowOwnerGUID` / `orderInList` match what the file implies
- (child rows only) an embedding is actually present and the right length
- nothing extra is sitting in the tables that no longer has a local file
  behind it (orphaned rows from before a section-count change)

Read-only throughout — this notebook never writes to Supabase.


In [1]:
%pip install -q -r requirements.txt


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Imports & client setup

In [2]:
import os
import re
import json
import uuid
import resource
from pathlib import Path

from dotenv import load_dotenv
from supabase import create_client, Client

load_dotenv()


def require_env(name: str) -> str:
    value = os.environ.get(name, "").strip()
    if not value:
        raise RuntimeError(
            f"{name} is empty in your .env file. Open .env in the RAG11 folder "
            f"and paste your actual value in after '{name}='."
        )
    return value


SUPABASE_URL = require_env("PUBLIC_SUPABASE_URL")
SUPABASE_KEY = os.environ.get("SUPABASE_SERVICE_ROLE_KEY", "").strip() or require_env("PUBLIC_SUPABASE_ANON_KEY")
supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

OUTPUT_ROOT = Path(".") / "stage1_eda_output"
SOURCE_KEYS = ["source1", "source2", "source3"]
PARENT_TABLE = "rag11_chunks_parent_table"
CHILD_TABLE = "rag11_chunks_child_table"
EMBEDDING_DIM = 1024   # must match create_sql_tables.sql / stage1_2's EMBEDDING_MODEL

soft_fd_limit, _ = resource.getrlimit(resource.RLIMIT_NOFILE)
print(f"This kernel process's open-file limit: soft={soft_fd_limit}")
print("Clients ready. Supabase project:", SUPABASE_URL)


This kernel process's open-file limit: soft=1048576
Clients ready. Supabase project: https://czgrxgzdmodkkmbmraub.supabase.co


## Load local chunk files + recompute their expected row shape

Same file-parsing and deterministic-`uuid5` logic as
`stage1_2_eda_load_chunks.ipynb`, duplicated here so this notebook is
self-contained and can be run independently, any time, without re-running
the loader.

In [3]:
PARENT_FILE_RE = re.compile(r"^parent_chunk-(\d+)\.json$")
CHILD_FILE_RE = re.compile(r"^child_chunk-parent(\d+)-chunk(\d+)\.json$")


def load_parent_files(source_key: str) -> list[tuple[int, dict]]:
    out = []
    for f in (OUTPUT_ROOT / source_key).glob("parent_chunk-*.json"):
        m = PARENT_FILE_RE.match(f.name)
        if not m:
            continue
        out.append((int(m.group(1)), json.loads(f.read_text(encoding="utf-8"))))
    out.sort(key=lambda t: t[0])
    return out


def load_child_files(source_key: str) -> list[tuple[int, int, dict]]:
    out = []
    for f in (OUTPUT_ROOT / source_key).glob("child_chunk-*.json"):
        m = CHILD_FILE_RE.match(f.name)
        if not m:
            continue
        out.append((int(m.group(1)), int(m.group(2)), json.loads(f.read_text(encoding="utf-8"))))
    out.sort(key=lambda t: (t[0], t[1]))
    return out


parents_by_source = {k: load_parent_files(k) for k in SOURCE_KEYS}
children_by_source = {k: load_child_files(k) for k in SOURCE_KEYS}
for k in SOURCE_KEYS:
    print(f"[{k}] {len(parents_by_source[k])} local parent file(s), "
          f"{len(children_by_source[k])} local child file(s)")

RAG11_UUID_NAMESPACE = uuid.uuid5(uuid.NAMESPACE_DNS, "rag11.nutrition.poc")


def deterministic_uuid(business_key: str) -> str:
    return str(uuid.uuid5(RAG11_UUID_NAMESPACE, business_key))


# expected_parent[rowGUID] = (rowOwnerGUID, orderInList, rowJSON-from-file)
expected_parent = {}
for source_key in SOURCE_KEYS:
    for order, data in parents_by_source[source_key]:
        guid = deterministic_uuid(f"parent:{data['parent_id']}")
        expected_parent[guid] = (source_key, order, data)

# expected_child[rowGUID] = (rowOwnerGUID, rowParentGUID, orderInList, rowJSON-from-file)
expected_child = {}
for source_key in SOURCE_KEYS:
    for _p_order, c_order, data in children_by_source[source_key]:
        guid = deterministic_uuid(f"child:{data['child_id']}")
        parent_guid = deterministic_uuid(f"parent:{data['parent_id']}")
        expected_child[guid] = (source_key, parent_guid, c_order, data)

print(f"\nExpected from local files: {len(expected_parent)} parent row(s), "
      f"{len(expected_child)} child row(s)")


[source1] 133 local parent file(s), 1083 local child file(s)
[source2] 426 local parent file(s), 2361 local child file(s)
[source3] 91 local parent file(s), 1491 local child file(s)

Expected from local files: 650 parent row(s), 4935 child row(s)


## Fetch every row from Supabase (paginated)

`select()` is capped per request, so this pages through with `.range()`
until a page comes back short — works the same whether the table has a
few hundred rows or a few hundred thousand.

In [4]:
def fetch_all_rows(table_name: str, columns: str, page_size: int = 1000) -> dict:
    """Return {rowGUID: row_dict} for every row in `table_name`."""
    rows_by_guid = {}
    start = 0
    while True:
        resp = (
            supabase.table(table_name)
            .select(columns)
            .range(start, start + page_size - 1)
            .execute()
        )
        page = resp.data
        for row in page:
            rows_by_guid[row["rowGUID"]] = row
        if len(page) < page_size:
            break
        start += page_size
    return rows_by_guid


print("Fetching all parent rows from Supabase...")
db_parent = fetch_all_rows(PARENT_TABLE, '"rowGUID","rowOwnerGUID","orderInList","rowJSON"')
print(f"  -> {len(db_parent)} row(s) in {PARENT_TABLE}")

print("Fetching all child rows from Supabase (including embeddings)...")
db_child = fetch_all_rows(
    CHILD_TABLE,
    '"rowGUID","rowOwnerGUID","rowParentGUID","orderInList","rowJSON","embedding"',
)
print(f"  -> {len(db_child)} row(s) in {CHILD_TABLE}")


Fetching all parent rows from Supabase...
  -> 650 row(s) in rag11_chunks_parent_table
Fetching all child rows from Supabase (including embeddings)...
  -> 4935 row(s) in rag11_chunks_child_table


## Compare — parent rows

In [5]:
def embedding_length(value) -> int:
    """pgvector comes back over PostgREST as either a JSON list of floats or
    a "[0.1,0.2,...]" string depending on client/library versions -- handle
    both so this check doesn't depend on which one you have installed."""
    if value is None:
        return 0
    if isinstance(value, list):
        return len(value)
    if isinstance(value, str):
        return value.count(",") + 1 if value.strip("[]") else 0
    return 0


parent_missing = []      # local file exists, no row in Supabase
parent_mismatched = []   # row exists, but content differs from the local file
parent_ok = 0

for guid, (owner, order, local_json) in expected_parent.items():
    row = db_parent.get(guid)
    if row is None:
        parent_missing.append((owner, order, local_json.get("parent_id")))
        continue
    if row["rowJSON"] != local_json or row["rowOwnerGUID"] != owner or row["orderInList"] != order:
        parent_mismatched.append((owner, order, local_json.get("parent_id")))
        continue
    parent_ok += 1

parent_orphaned = [guid for guid in db_parent if guid not in expected_parent]

print(f"Parent rows -- OK: {parent_ok}, missing: {len(parent_missing)}, "
      f"mismatched: {len(parent_mismatched)}, orphaned in DB: {len(parent_orphaned)}")

for label, items in [("missing", parent_missing), ("mismatched", parent_mismatched)]:
    if items:
        print(f"\n  First {min(5, len(items))} {label} parent row(s):")
        for owner, order, parent_id in items[:5]:
            print(f"    [{owner} #{order}] {parent_id}")


Parent rows -- OK: 650, missing: 0, mismatched: 0, orphaned in DB: 0


## Compare — child rows (+ embedding presence/dimension)

In [6]:
child_missing = []
child_mismatched = []
child_missing_embedding = []
child_ok = 0

for guid, (owner, parent_guid, order, local_json) in expected_child.items():
    row = db_child.get(guid)
    if row is None:
        child_missing.append((owner, order, local_json.get("child_id")))
        continue
    content_matches = (
        row["rowJSON"] == local_json
        and row["rowOwnerGUID"] == owner
        and row["rowParentGUID"] == parent_guid
        and row["orderInList"] == order
    )
    if not content_matches:
        child_mismatched.append((owner, order, local_json.get("child_id")))
        continue
    if embedding_length(row.get("embedding")) != EMBEDDING_DIM:
        child_missing_embedding.append((owner, order, local_json.get("child_id")))
        continue
    child_ok += 1

child_orphaned = [guid for guid in db_child if guid not in expected_child]

print(f"Child rows -- OK: {child_ok}, missing: {len(child_missing)}, "
      f"mismatched: {len(child_mismatched)}, bad/missing embedding: {len(child_missing_embedding)}, "
      f"orphaned in DB: {len(child_orphaned)}")

for label, items in [
    ("missing", child_missing),
    ("mismatched", child_mismatched),
    ("with a bad/missing embedding", child_missing_embedding),
]:
    if items:
        print(f"\n  First {min(5, len(items))} child row(s) {label}:")
        for owner, order, child_id in items[:5]:
            print(f"    [{owner} #{order}] {child_id}")


Child rows -- OK: 4935, missing: 0, mismatched: 0, bad/missing embedding: 0, orphaned in DB: 0


## Row-count summary by source

Cross-checks local file counts against Supabase counts per
`rowOwnerGUID`, which is usually the fastest way to spot a stale/orphaned
source after re-tuning stage1_1 (e.g. Source 2 shrinking from 426 to 77
sections after the running-header fix).

In [7]:
from collections import Counter

local_parent_counts = Counter(owner for owner, _, _ in expected_parent.values())
db_parent_counts = Counter(row["rowOwnerGUID"] for row in db_parent.values())
local_child_counts = Counter(owner for owner, _, _, _ in expected_child.values())
db_child_counts = Counter(row["rowOwnerGUID"] for row in db_child.values())

print(f"{'source':<10} {'local parents':>14} {'db parents':>11} {'local children':>15} {'db children':>12}")
for source_key in SOURCE_KEYS:
    flag = "" if (local_parent_counts[source_key] == db_parent_counts[source_key]
                  and local_child_counts[source_key] == db_child_counts[source_key]) else "  <-- mismatch"
    print(f"{source_key:<10} {local_parent_counts[source_key]:>14} {db_parent_counts[source_key]:>11} "
          f"{local_child_counts[source_key]:>15} {db_child_counts[source_key]:>12}{flag}")


source      local parents  db parents  local children  db children
source1               133         133            1083         1083
source2               426         426            2361         2361
source3                91          91            1491         1491


## Smoke-test vector search

Confirms `match_rag11_child_chunks` (the RPC from `create_sql_tables.sql`)
actually returns results end to end, using a real embedding already
stored in the table (no Voyage API call needed here).

In [8]:
sample_child = next(iter(db_child.values()), None)
if sample_child is None:
    print("No child rows in the table yet -- nothing to smoke-test.")
else:
    query_embedding = sample_child["embedding"]
    resp = supabase.rpc("match_rag11_child_chunks", {
        "query_embedding": query_embedding,
        "match_count": 3,
    }).execute()
    print("Smoke-test match_rag11_child_chunks (querying with one child's own embedding):")
    for row in resp.data:
        preview = row["rowJSON"]["text"][:80].replace("\n", " ")
        print(f"  [{row['rowOwnerGUID']} #{row['orderInList']}] "
              f"dist={row['cosine_distance']:.4f}  {preview}...")


Smoke-test match_rag11_child_chunks (querying with one child's own embedding):
  [source1 #29] dist=0.0000  [Source: human-nutrition-text.pdf | Section: Chapter 8. Energy | Pages 493-554] ...
  [source1 #53] dist=0.2981  [Source: human-nutrition-text.pdf | Section: Chapter 8. Energy | Pages 493-554] ...
  [source1 #43] dist=0.3146  [Source: human-nutrition-text.pdf | Section: Chapter 8. Energy | Pages 493-554] ...


## Overall verdict

In [9]:
total_issues = (
    len(parent_missing) + len(parent_mismatched) + len(parent_orphaned)
    + len(child_missing) + len(child_mismatched) + len(child_missing_embedding) + len(child_orphaned)
)

if total_issues == 0:
    print("PASS -- every local chunk file matches its row in Supabase exactly, "
          "and every child row has a valid embedding. No orphaned rows either.")
else:
    print(f"FAIL -- {total_issues} issue(s) found across the checks above.")
    if parent_orphaned or child_orphaned:
        print(f"  {len(parent_orphaned)} orphaned parent row(s) + {len(child_orphaned)} orphaned "
              f"child row(s) in Supabase have no matching local file -- likely leftovers from "
              f"before a section-count change. See delete_chunks_data.sql to clear them, then "
              f"re-run stage1_2_eda_load_chunks.ipynb.")
    if child_missing or parent_missing:
        print("  Some local chunks never made it into Supabase -- re-run "
              "stage1_2_eda_load_chunks.ipynb (its checkpointing will skip what already succeeded).")


PASS -- every local chunk file matches its row in Supabase exactly, and every child row has a valid embedding. No orphaned rows either.
